# LightshowAI XAS notebook

This notebook shows how to fetch a crystal structure from the Materials Project, download the required LightshowAI model checkpoints automatically, and run a simple XAS prediction for a chosen absorbing element and theory combination.

## Before you run it

1. Install the required Python packages in your environment.
2. Create a Materials Project API key.
3. Set that key in the configuration cell below, or export it as an environment variable named `MP_API_KEY`.

## How to generate a Materials Project API key

1. Create or sign in to your Materials Project account.
2. Open your dashboard or API settings page.
3. Generate a new API key.
4. Copy the key and paste it into the configuration cell, or set it in your shell with `export MP_API_KEY=...` before starting Jupyter.

## Expected outputs

- A downloaded `model_checkpoints/` folder in the notebook working directory.
- A structure summary for the selected Materials Project material.
- A dictionary-like prediction output for the selected absorbing element.


In [1]:
from __future__ import annotations

# Standard-library imports used for dynamic imports, JSON formatting,
# environment variables, and filesystem path handling.
import importlib
import json
import os
import sys
from pathlib import Path
from typing import Any

# Materials Project client used to fetch crystal structures by material ID.
from mp_api.client import MPRester
from pymatgen.core import Structure


/home/sairam/miniforge3/envs/LightshowAI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def add_repo_to_path(repo_path: str) -> None:
    """Add a repository path to sys.path so local modules can be imported.

    This is optional in this notebook, but can be useful if you have a local
    clone of a project and want to import Python modules from it.
    """
    repo = str(Path(repo_path).resolve())
    if repo not in sys.path:
        sys.path.insert(0, repo)


def fetch_structure(material_id: str, api_key: str) -> Structure:
    """Fetch a pymatgen Structure object from Materials Project.

    Parameters
    ----------
    material_id
        A Materials Project ID such as 'mp-390'.
    api_key
        Your Materials Project API key.

    Notes
    -----
    You can either pass the key directly or store it in the environment as
    MP_API_KEY and read it in the configuration cell below.
    """
    if not api_key:
        raise ValueError(
            "No Materials Project API key found. Set MP_API_KEY or assign MP_API_KEY in the config cell."
        )

    with MPRester(api_key) as mpr:
        structure = mpr.get_structure_by_material_id(material_id)

    if structure is None:
        raise RuntimeError(f"No structure returned for material ID: {material_id}")

    return structure


def import_model_module(module_name: str):
    """Import a module by name. Useful for modular notebook workflows."""
    return importlib.import_module(module_name)


def get_callable(module, fn_name: str):
    """Safely retrieve a callable from a Python module."""
    try:
        fn = getattr(module, fn_name)
    except AttributeError as exc:
        raise AttributeError(
            f"Module '{module.__name__}' does not define function '{fn_name}'"
        ) from exc

    if not callable(fn):
        raise TypeError(
            f"Attribute '{fn_name}' in module '{module.__name__}' is not callable"
        )
    return fn


def structure_summary(structure: Structure) -> dict[str, Any]:
    """Return a JSON-friendly summary of a pymatgen structure.

    This is helpful for quick inspection in notebooks because the full
    Structure object contains richer methods and metadata than a simple print.
    """
    return {
        "formula": structure.composition.reduced_formula,
        "num_sites": len(structure),
        "lattice": structure.lattice.as_dict(),
        "species": [str(site.specie) for site in structure],
        "cart_coords": [list(map(float, site.coords)) for site in structure],
        "frac_coords": [list(map(float, site.frac_coords)) for site in structure],
    }


def to_jsonable(obj: Any) -> Any:
    """Convert common Python objects to values that json.dumps can print."""
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if hasattr(obj, "as_dict"):
        try:
            return obj.as_dict()
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {
                k: to_jsonable(v)
                for k, v in vars(obj).items()
                if not k.startswith("_")
            }
        except Exception:
            pass
    return repr(obj)


def print_block(title: str, obj: Any, pretty: bool = False) -> None:
    """Pretty-print structured content with a section heading."""
    print(f"\n=== {title} ===")
    payload = to_jsonable(obj)
    if pretty:
        print(json.dumps(payload, indent=2, sort_keys=False))
    else:
        print(json.dumps(payload))


## Configure your input material and API key

Update the material ID and provide your Materials Project API key.

### Options for the API key

- Paste it directly into `MP_API_KEY` below.
- Or keep `MP_API_KEY = os.getenv("MP_API_KEY", "")` so the notebook reads it from your environment.

Using an environment variable is safer because it avoids saving the key inside the notebook file.


In [ ]:
# Materials Project material identifier. Examples look like mp-390, mp-149, etc.
MATERIAL_ID = "mp-390"

# Preferred option: read your API key from an environment variable.
# In a terminal before launching Jupyter, you can run:
#   export MP_API_KEY="your_api_key_here"
# You can also paste the key directly as a string, but avoid committing it to git.
MP_API_KEY = os.getenv("MP_API_KEY", "")

# Set PRETTY=True for readable JSON-like output in the notebook.
PRETTY = True

print(f"Fetching structure for {MATERIAL_ID} from Materials Project...")
structure = fetch_structure(MATERIAL_ID, MP_API_KEY)
print_block("STRUCTURE SUMMARY", structure_summary(structure), pretty=PRETTY)


## Download model checkpoints and define the prediction code

This cell does three things:

1. Creates a local `model_checkpoints/` folder if it does not exist.
2. Downloads the LightshowAI checkpoint files from GitHub if they are missing.
3. Defines the helper classes used to featurize the structure and run prediction.

The downloads happen only once unless you set `overwrite=True` in `ensure_model_checkpoints()`.


In [ ]:
import pathlib
import urllib.request
from functools import cache
from typing import List

import numpy as np
import torch
from lightning import LightningModule
from matgl import load_model
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.compute import (
    compute_pair_vector_and_distance,
    compute_theta_and_phi,
    create_line_graph,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Structure as PymatgenStructure
from torch import nn

# Base location of the published LightshowAI checkpoint files.
# These are downloaded lazily into the local notebook working directory.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AI-multimodal/LightshowAI/main/model_checkpoints"

# Notebook-safe working directory.
# If you started Jupyter in a project folder, the model_checkpoints directory
# will be created there.
PARENT_DIRECTORY = pathlib.Path.cwd().resolve()
MODEL_CHECKPOINTS_PATH = PARENT_DIRECTORY / "model_checkpoints"
XASBLOCKS_PATH = MODEL_CHECKPOINTS_PATH / "xasblock" / "v1.1.1"
M3GNET_PATH = MODEL_CHECKPOINTS_PATH / "M3GNet-MP-2021.2.8-PES"

# Available XAS block checkpoints released in the LightshowAI repository.
# The file naming convention is <ELEMENT>_<THEORY>.ckpt
XASBLOCK_FILES = [
    "Co_FEFF.ckpt",
    "Cr_FEFF.ckpt",
    "Cu_FEFF.ckpt",
    "Cu_VASP.ckpt",
    "Fe_FEFF.ckpt",
    "Mn_FEFF.ckpt",
    "Ni_FEFF.ckpt",
    "Ti_FEFF.ckpt",
    "Ti_VASP.ckpt",
    "V_FEFF.ckpt",
]

# Files required to load the M3GNet backbone used for structure featurization.
M3GNET_FILES = [
    "LICENSE",
    "README.md",
    "model.json",
    "model.pt",
    "state.pt",
]


def _download_file(url: str, destination: pathlib.Path, overwrite: bool = False):
    """Download one file if it is missing locally."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        return
    print(f"Downloading {destination.name} ...")
    urllib.request.urlretrieve(url, destination)


def ensure_model_checkpoints(overwrite: bool = False):
    """Create model folders and download required checkpoint files.

    Run this once at the beginning of a session. It is safe to call multiple
    times because existing files are skipped unless overwrite=True.
    """
    XASBLOCKS_PATH.mkdir(parents=True, exist_ok=True)
    M3GNET_PATH.mkdir(parents=True, exist_ok=True)

    for filename in XASBLOCK_FILES:
        url = f"{GITHUB_RAW_BASE}/xasblock/v1.1.1/{filename}"
        dest = XASBLOCKS_PATH / filename
        _download_file(url, dest, overwrite=overwrite)

    for filename in M3GNET_FILES:
        url = f"{GITHUB_RAW_BASE}/M3GNet-MP-2021.2.8-PES/{filename}"
        dest = M3GNET_PATH / filename
        _download_file(url, dest, overwrite=overwrite)


# Ensure checkpoints are present before any model code runs.
ensure_model_checkpoints()

# Useful for checking which element/theory combinations are available locally.
AVAILABLE_COMBINATIONS = sorted(f.stem for f in XASBLOCKS_PATH.glob("*.ckpt"))
print("Available checkpoint combinations:", AVAILABLE_COMBINATIONS)


class XASBlock(nn.Sequential):
    """Simple feed-forward block used by the released checkpoint."""
    DROPOUT = 0.5

    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int):
        dims = [input_dim] + hidden_dims + [output_dim]
        layers = []
        for i, (w1, w2) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(w1, w2))
            if i < len(dims) - 2:
                layers.append(nn.BatchNorm1d(w2))
                layers.append(nn.SiLU())
                layers.append(nn.Dropout(self.DROPOUT))
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class XASBlockModule(LightningModule):
    """Lightning wrapper around the XAS block checkpoint."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    @classmethod
    def load(
        cls,
        element: str,
        spectroscopy_type: str,
        pattern=XASBLOCKS_PATH / "{element}_{type}.ckpt",
    ):
        # Make sure files are present before attempting to load a checkpoint.
        ensure_model_checkpoints()
        pattern = str(pattern)
        path = pattern.format(element=element, type=spectroscopy_type)

        if not pathlib.Path(path).exists():
            raise FileNotFoundError(
                f"Checkpoint not found: {path}\n"
                f"Available combinations: {AVAILABLE_COMBINATIONS}"
            )

        # These dimensions match the published v1.1.1 checkpoint layout.
        model = XASBlock(
            input_dim=64,
            hidden_dims=[500, 500, 550],
            output_dim=141,
        )
        module = cls.load_from_checkpoint(checkpoint_path=path, model=model)
        return module


class M3GNetFeaturizer:
    """Convert a pymatgen Structure into learned node features using M3GNet."""
    def __init__(self, model=None, n_blocks=None):
        self.model = model or M3GNetFeaturizer._load_m3gnet()
        self.model.eval()
        self.n_blocks = n_blocks or self.model.n_blocks

    def featurize(
        self,
        structure: PymatgenStructure,
    ):
        graph_converter = Structure2Graph(
            self.model.element_types, self.model.cutoff
        )
        g, state_attr = graph_converter.get_graph(structure)

        node_types = g.ndata["node_type"]
        bond_vec, bond_dist = compute_pair_vector_and_distance(g)

        g.edata["bond_vec"] = bond_vec.to(g.device)
        g.edata["bond_dist"] = bond_dist.to(g.device)

        with torch.no_grad():
            expanded_dists = self.model.bond_expansion(g.edata["bond_dist"])

            l_g = create_line_graph(g, self.model.threebody_cutoff)

            l_g.apply_edges(compute_theta_and_phi)
            g.edata["rbf"] = expanded_dists
            three_body_basis = self.model.basis_expansion(l_g)
            three_body_cutoff = polynomial_cutoff(
                g.edata["bond_dist"], self.model.threebody_cutoff
            )
            node_feat, edge_feat, state_feat = self.model.embedding(
                node_types, g.edata["rbf"], state_attr
            )

            for i in range(self.n_blocks):
                edge_feat = self.model.three_body_interactions[i](
                    g,
                    l_g,
                    three_body_basis,
                    three_body_cutoff,
                    node_feat,
                    edge_feat,
                )
                edge_feat, node_feat, state_feat = self.model.graph_layers[i](
                    g, edge_feat, node_feat, state_feat
                )

        # Move features to CPU before converting to NumPy for notebook use.
        res = np.array(node_feat.detach().cpu().numpy())
        return res

    @cache
    @staticmethod
    def _load_m3gnet(path=M3GNET_PATH):
        """Load and cache the M3GNet model so repeated predictions are faster."""
        ensure_model_checkpoints()
        model = load_model(path).model
        model.eval()
        return model


class XASModel:
    """High-level wrapper that featurizes a structure and predicts a spectrum."""
    featurizer = M3GNetFeaturizer()

    def __init__(self, element: str, spectroscopy_type: str):
        self.element = element
        self.spectroscopy_type = spectroscopy_type
        self.model = XASBlockModule.load(
            element=element, spectroscopy_type=spectroscopy_type
        )
        self.model.eval()

    def _get_feature(self, structure: PymatgenStructure):
        return self.featurizer.featurize(structure)

    def predict(
        self,
        structure: PymatgenStructure,
    ):
        with torch.no_grad():
            feature = self._get_feature(structure)

            # The released notebook code scales the feature vector by 1000
            # before feeding it into the XAS block.
            feature = feature * 1000.0

            device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
            feature = torch.tensor(feature, device=device)
            spectrum = self.model(feature)

        spectrum = spectrum.detach().cpu().numpy().squeeze()
        return spectrum


def predict(structure, absorbing_site, spectroscopy_type):
    """Predict spectra for a given absorbing element in the input structure.

    Parameters
    ----------
    structure
        pymatgen Structure object.
    absorbing_site
        Element symbol such as 'Ti' or 'Fe'.
    spectroscopy_type
        Theory label that matches a downloaded checkpoint, such as 'FEFF' or 'VASP'.
    """
    site_idxs = [
        ii
        for ii, site in enumerate(structure.sites)
        if site.specie.symbol == absorbing_site
    ]
    if len(site_idxs) == 0:
        raise ValueError(
            f"element {absorbing_site} not found in provided structure"
        )

    spec = XASModel(
        element=absorbing_site, spectroscopy_type=spectroscopy_type
    ).predict(structure)

    result = {ii: spec[ii] for ii in site_idxs}
    return result


## Run a prediction

Choose an absorbing element and theory that exist in `AVAILABLE_COMBINATIONS`.

Examples from the downloaded checkpoints include `Ti_VASP`, `Ti_FEFF`, `Fe_FEFF`, and `Cu_VASP`.


In [ ]:
# Example prediction: titanium using the VASP-trained checkpoint.
# Make sure the element/theory pair exists in AVAILABLE_COMBINATIONS.
element = "Ti"
theory = "VASP"

output = predict(structure, element, theory)
print_block("PREDICTION OUTPUT", output, pretty=True)


## Troubleshooting

- **`No Materials Project API key found`**: generate a key in your Materials Project account and set `MP_API_KEY`.
- **Checkpoint not found**: print `AVAILABLE_COMBINATIONS` and choose one of the listed element/theory pairs.
- **Download errors**: verify you have internet access from the notebook environment.
- **Import errors**: install missing packages such as `mp-api`, `pymatgen`, `matgl`, `lightning`, and `torch`.
